In [1]:
import cudf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv("./data/train.csv", parse_dates=['pickup_datetime', 'dropoff_datetime'])

In [2]:
def printd(data, step=8):
    """
    Prints a pandas DataFrame in vertical markdown chunks to prevent horizontal overflow.

    This function slices a wide DataFrame into manageable blocks of columns and prints 
    each block as a formatted markdown table. It is particularly useful for inspecting 
    datasets with hundreds of features in terminal-based environments or notebook 
    interfaces where horizontal scrolling is restricted.

    Args:
        data (pd.DataFrame): The pandas DataFrame to be printed.
        step (int, optional): The maximum number of columns to display per chunk. 
                              Defaults to 8.
    """
    for i in range(0, len(data.columns), step):
        chunk = data.iloc[:, i:i+step]
        print(chunk.to_markdown())
        print("\n")

f5_rows = data.head()
printd(f5_rows, step=10)

|    | id        |   vendor_id | pickup_datetime     | dropoff_datetime    |   passenger_count |   pickup_longitude |   pickup_latitude |   dropoff_longitude |   dropoff_latitude | store_and_fwd_flag   |
|---:|:----------|------------:|:--------------------|:--------------------|------------------:|-------------------:|------------------:|--------------------:|-------------------:|:---------------------|
|  0 | id2875421 |           2 | 2016-03-14 17:24:55 | 2016-03-14 17:32:30 |                 1 |           -73.9822 |           40.7679 |            -73.9646 |            40.7656 | N                    |
|  1 | id2377394 |           1 | 2016-06-12 00:43:35 | 2016-06-12 00:54:38 |                 1 |           -73.9804 |           40.7386 |            -73.9995 |            40.7312 | N                    |
|  2 | id3858529 |           2 | 2016-01-19 11:35:24 | 2016-01-19 12:10:48 |                 1 |           -73.979  |           40.7639 |            -74.0053 |            40.7101 | N  

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458644 entries, 0 to 1458643
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   id                  1458644 non-null  object        
 1   vendor_id           1458644 non-null  int64         
 2   pickup_datetime     1458644 non-null  datetime64[ns]
 3   dropoff_datetime    1458644 non-null  datetime64[ns]
 4   passenger_count     1458644 non-null  int64         
 5   pickup_longitude    1458644 non-null  float64       
 6   pickup_latitude     1458644 non-null  float64       
 7   dropoff_longitude   1458644 non-null  float64       
 8   dropoff_latitude    1458644 non-null  float64       
 9   store_and_fwd_flag  1458644 non-null  object        
 10  trip_duration       1458644 non-null  int64         
dtypes: datetime64[ns](2), float64(4), int64(3), object(2)
memory usage: 122.4+ MB


In [4]:
printd( data.describe(), 9 )

|       |   vendor_id | pickup_datetime               | dropoff_datetime              |   passenger_count |   pickup_longitude |   pickup_latitude |   dropoff_longitude |   dropoff_latitude |   trip_duration |
|:------|------------:|:------------------------------|:------------------------------|------------------:|-------------------:|------------------:|--------------------:|-------------------:|----------------:|
| count | 1.45864e+06 | 1458644                       | 1458644                       |       1.45864e+06 |        1.45864e+06 |       1.45864e+06 |         1.45864e+06 |        1.45864e+06 |     1.45864e+06 |
| mean  | 1.53495     | 2016-04-01 10:10:24.940037120 | 2016-04-01 10:26:24.432310528 |       1.66453     |      -73.9735      |      40.7509      |       -73.9734      |       40.7518      |   959.492       |
| min   | 1           | 2016-01-01 00:00:17           | 2016-01-01 00:03:31           |       0           |     -121.933       |      34.3597      |      -121.9

In [5]:
columns = list(data.columns[5:9])
columns

['pickup_longitude',
 'pickup_latitude',
 'dropoff_longitude',
 'dropoff_latitude']

In [6]:
import cupy as cp
from cuml.cluster import kmeans

def preprocess(data, columns):
    ext_data = cudf.from_pandas(data)

    def bin_arranger(data, columns):
        new_binned_cols = {}
        
        for col in columns:
            col_data = data[col]
            mean = col_data.mean()
            std = col_data.std()
            c_min = col_data.min()
            c_max = col_data.max()
            
            min_range = abs(mean - c_min) / 4
            max_range = abs(c_max - mean) / 4
            
            raw_bins = [
                c_min, mean - min_range*3, mean - min_range*2, mean - min_range, 
                mean - std*3, mean - std*2, mean - std, mean, mean + std, mean + std*2, mean + std*3, 
                mean + max_range, mean + max_range*2, mean + max_range*3, c_max
            ]
            
            bins = sorted(list(set(raw_bins)))
            bins[0] = -np.inf
            bins[-1] = np.inf
            
            parts = col.split("_")
            name = f"{parts[0][0]}_{parts[1][:2]}_binned"
            
            binned_cat = cudf.cut(col_data, bins=bins, include_lowest=True)
            new_binned_cols[name] = binned_cat.cat.codes

        return cudf.concat([data, cudf.DataFrame(new_binned_cols)], axis=1)

    ext_data = bin_arranger(ext_data, columns)


    ext_data['hour'] = ext_data['pickup_datetime'].dt.hour
    ext_data['day_of_week'] = ext_data['pickup_datetime'].dt.dayofweek
    ext_data['month'] = ext_data['pickup_datetime'].dt.month
    ext_data['weekend'] = ext_data['day_of_week'].isin([5,6])

    ext_data["vendor"] = ext_data["vendor_id"] == 2

    ext_data["store_and_fwd_flag_bin"] = ext_data["store_and_fwd_flag"] == "Y"

    lon1 = ext_data['pickup_longitude']*cp.pi/180 
    lon2 = ext_data['dropoff_longitude']*cp.pi/180 
    lat1 = ext_data['pickup_latitude']*cp.pi/180 
    lat2 = ext_data['dropoff_latitude']*cp.pi/180 

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = cp.sin(dlat/2)**2 + cp.cos(lat1) * cp.cos(lat2) * cp.sin(dlon/2)**2
    c = 2*cp.arctan2(cp.sqrt(a), cp.sqrt(1-a))

    ext_data["distance"] = c*6371

    ml_data = ext_data.drop(["id", "vendor_id", "pickup_datetime", "dropoff_datetime", "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude", "store_and_fwd_flag"], axis=1, errors="ignore")
    try:
        ml_data = ml_data[ml_data['trip_duration'] < ml_data["trip_duration"].quantile(0.998)]
    except KeyError:
        pass

    return ext_data, ml_data

In [7]:
ext_data, ml_data = preprocess(data, columns)

In [9]:
import matplotlib.pyplot as plt

# Adjusted bounding box strictly for the main NYC area 
# to ensure the map isn't zoomed out too far
nyc_bounds = ([-74.05, -73.75], [40.60, 40.90])

# Safely transfer only the needed coordinates to the CPU
x = ext_data['pickup_longitude'].to_numpy()
y = ext_data['pickup_latitude'].to_numpy()

plt.figure(figsize=(10, 10))
# Using hist2d creates a beautiful density heatmap
plt.hist2d(x, y, bins=300, range=nyc_bounds, cmap='inferno')
plt.colorbar(label='Number of Pickups')
plt.title('NYC Taxi Pickup Density')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
X = ml_data.drop("trip_duration", axis=1)
y = ml_data["trip_duration"]

In [ ]:
printd(ext_data.to_pandas().head(), 11)

In [ ]:
y[y < y.quantile(.998)].describe().round(3)

y_labels = y.quantile([0, 0.1, 0.25, 0.5, .75, .9, 1]).to_pandas().tolist()
y_labels[0] = y_labels[0] - .001
y_labels[-1] = y_labels[-1] + .001

y_labels = cudf.cut(y, bins=y_labels, include_lowest=True).cat.codes

In [ ]:
from cuml.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y_labels)

y_train = cp.log1p(y_train)
y_test = cp.log1p(y_test)

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# 1. Initialize the XGBoost Regressor with GPU settings
xgb_model = XGBRegressor(
    # --- The most critical GPU parameters ---
    device='cuda',            # Forces XGBoost to use the GPU (Modern XGBoost 2.0+ API)
    tree_method='hist',       # The fastest algorithm for GPU training
    
    # --- Standard Hyperparameters (Adjust these later) ---
    n_estimators=1000,        # Maximum number of trees to build
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds = 50,
    random_state=42
)

# 2. Train the model directly on the cuDF splits
print("Starting GPU Training...")
xgb_model.fit(
    X_train, 
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)], # Monitor both sets for overfitting
    verbose=50 # Print the score every 100 trees
)
print("Training Complete!")

In [ ]:
test_df = pd.read_csv("./data/test.csv", parse_dates=["pickup_datetime"])
test_df = test_df.set_index("id")

In [ ]:
printd(test_df.head(), 11)

In [ ]:
ext_test_data, ml_test_data = preprocess(test_df, columns)

In [ ]:
printd( ml_test_data.head().to_pandas(), 13)

In [ ]:
preds = xgb_model.predict(ml_test_data)
preds = np.expm1(preds)

In [ ]:
submission = cudf.DataFrame({'id': ml_test_data.index, 'trip_duration': preds})

In [ ]:
submission.to_csv('submission_xgb.csv', index=False)